In [1]:
import pandas as pd
import os
from openai import OpenAI

In [2]:
api_key = ""
with open ("api kety.txt" , "r") as f:
    api_key  = f.read()

In [3]:
client = OpenAI(api_key = api_key)

In [4]:
def get_median_length(df, column_name='human_answer'):
    """Calculates and returns the median string length of a specific DataFrame column."""
    return int(df[column_name].astype(str).apply(len).median())

In [5]:
def generate_ai_answers(input_path, output_path, domain):
    print(f"--- Starting AI generation for {domain} ---")
    
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    df_human = pd.read_csv(input_path, encoding='utf-8-sig')
    ai_rows = []
    
    # Calculate the median length using the separate function
    median_length = get_median_length(df_human)
    print(f"[{domain}] Target median length for AI: ~{median_length} characters.")
    
    # UPDATED SYSTEM PROMPT: Tone directives removed, length injected via f-string
    system_prompt = (
        f"You are a highly experienced and professional doctor specializing in {domain}. "
        "You are the patient's primary treating physician. "
        "CRITICAL RULES:\n"
        "1. ONLY RESPOND IN CHINESE. Do not use English.\n"
        "2. NO BULLET POINTS OR LISTS. Respond strictly in continuous sentences and paragraphs. Do not use numbered lists or bullet points under any circumstances.\n"
        f"3. MATCH LENGTH: Your response must be approximately {median_length} Chinese characters long to match standard consultation lengths.\n"
        "4. NEVER tell the patient to 'consult your doctor' or 'seek medical attention'. YOU ARE their doctor.\n"
        "5. Assume full authority and responsibility for the case.\n"
        "6. NO AI disclaimers or robotic introductions (e.g., 'As an AI...')."
    )
    
    total_rows = len(df_human)
    
    for idx, row in df_human.iterrows():
        department = str(row['department'])
        title = str(row['title'])
        question = str(row['question'])
        
        patient_query = f"Title/Context: {title}\nQuestion: {question}"
        
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": patient_query}
                ],
                temperature=0.8 
            )
            ai_answer = response.choices[0].message.content.strip()
            
            # Print the Question and Answer in Chinese for each row
            print(f"\nQ: {idx} done creating")
            print(f"A: {idx} answered")
            print("-" * 50)
            
        except Exception as e:
            print(f"\nAPI Error at index {idx}: {e}")
            ai_answer = ""
            
        ai_rows.append({
            "department": department,
            "title": title,
            "question": question,
            "ai_answer": ai_answer
        })
        
        print(f"\r[{domain}] Progress: {idx + 1}/{total_rows} AI answers generated.", end="", flush=True)
        
    print() 
    df_ai = pd.DataFrame(ai_rows)
    df_ai.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"Successfully saved {len(df_ai)} AI responses to {output_path}\n")

In [6]:
if __name__ == "__main__":
    generate_ai_answers('Human datasets/Pediatrics 1000 Human.csv', 'Ai Datasets/Pediatrics 1000 AI.csv', 'Pediatrics')

--- Starting AI generation for Pediatrics ---
[Pediatrics] Target median length for AI: ~161 characters.

Q: 0 done creating
A: 0 answered
--------------------------------------------------
[Pediatrics] Progress: 1/1000 AI answers generated.
Q: 1 done creating
A: 1 answered
--------------------------------------------------
[Pediatrics] Progress: 2/1000 AI answers generated.

KeyboardInterrupt: 

In [7]:
with open('Human datasets/IM 1000 Human.csv', 'r' , encoding='utf-8-sig')  as f:
    print(f.readlines())

['department,title,question,human_answer\n', '普通内科,口里股大便味有胃痛吃了药没有好该怎么办？,长期口里撒气都有一股大便味，还伴发胃痛，怎么办？,应该是饮食不当，消化道菌群失调引起的消化道异常煮沸致使的口气较重，伴发胃痛等不适症状；考量不是胆结石；建议口服金双歧片可直接消化正常生理性细菌，调整肠道菌群平衡，救治由肠道菌群失调引来的急慢性腹泻、习惯性便秘、结肠炎、小儿厌食、消化不良等不适症状；留意歇息，多喝水，清淡饮食，不吃辛辣油腻刺激性食物，不吃极冷不易吸收的食物，多吃水果蔬菜，防寒保暖，胃痛病情很严重，建议患者及时救治，期望患者可以根据医生的意见对症救治。同时看重自身的护理培训，恰当饮食。\n', '肝病科,晚期肝硬化的症状表现在哪些方面,我的领导的父亲从发现肝硬化，就一直治疗，但是好像还在加重，我特别担心，想知道晚期肝硬化的症状有哪些表现？,男性肝硬化症状主要表现为：性欲缺乏、乳房发育、阳痿伴睾丸缩小以及前列腺缩小等。女性肝硬化的症状可表现为月经减少、月经失调、闭经、不育症、性欲减退、第二性征如会阴、乳房脂肪丧失及卵巢萎缩。\n', '内分泌科,甲状腺减退要长期服用药物吗,"超声描述:甲状腺多切面扫描，体积大，形态饱满，大小分别为:峡部前后径约4.3mm,左叶57x21X21mm,右叶61x23x20mm,轮廊清，腺体回声增粗，增强，分布不均匀，色彩多普勒血流显像:腺体内可见丰富的血流信号。超声提示:双侧甲状腺不均质肿大（结合临床）",病情分析：TSH促甲状腺激素明显升高，属于甲减的诊断敏感指标指导意见：一旦诊断为甲减，首先需要明确病因，然后针对性治疗。同时需要长期服用甲状腺素片，产后甲状腺炎要根据患者情况和确诊结果来选择治疗方法。想要得到病情缓解的话，患者还需要重视适当休息，饮食给予足够的营养和热量。\n', '肾内科,肾结石激光碎石有哪些后遗症,我今年岁数也不大啊，但是总是感觉怕冷，我浑身也没有力气干不了事情，这个毛病很久了，我害怕我的肾有毛病，请问肾结石激光碎石有哪些后遗症,"1.碎石后第一次小便可能呈深咖啡色,这是由于被粉碎的结石颗粒随尿道排出时,摩擦输尿管黏膜引起的少量出血,可不必紧张.到第二次,第三次颜色渐淡,至恢复正常.如果持续血尿,则应去医院复诊.2.碎石后如出现发烧,同时有尿频,尿急,尿痛等症状,

In [8]:
if __name__ == "__main__":
    generate_ai_answers('Human datasets/IM 1000 Human.csv', 'Ai Datasets/IM 1000 AI.csv', 'Internal Medicine')

--- Starting AI generation for Internal Medicine ---
[Internal Medicine] Target median length for AI: ~146 characters.

Q: 0 done creating
A: 0 answered
--------------------------------------------------
[Internal Medicine] Progress: 1/1000 AI answers generated.
Q: 1 done creating
A: 1 answered
--------------------------------------------------
[Internal Medicine] Progress: 2/1000 AI answers generated.
Q: 2 done creating
A: 2 answered
--------------------------------------------------
[Internal Medicine] Progress: 3/1000 AI answers generated.
Q: 3 done creating
A: 3 answered
--------------------------------------------------
[Internal Medicine] Progress: 4/1000 AI answers generated.
Q: 4 done creating
A: 4 answered
--------------------------------------------------
[Internal Medicine] Progress: 5/1000 AI answers generated.
Q: 5 done creating
A: 5 answered
--------------------------------------------------
[Internal Medicine] Progress: 6/1000 AI answers generated.
Q: 6 done creating
A: 